In [203]:
import pandas as pd
import numpy as np
import re
from typing import Dict
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [204]:
df = pd.read_json(
    "../data/threat_intel_dataset.jsonl",
    lines=True
)

df.head()

,instruction,input,output
0,You are a cybersecurity threat intelligence an...,CVE ID: CVE-2001-1481\nDescription: Xitami 2.4...,THREAT INTELLIGENCE REPORT\n\nSEVERITY: CRITIC...
1,You are a cybersecurity threat intelligence an...,CVE ID: CVE-1999-0426\nDescription: The defaul...,THREAT INTELLIGENCE REPORT\n\nSEVERITY: CRITIC...
2,You are a cybersecurity threat intelligence an...,CVE ID: CVE-1999-1324\nDescription: VAXstation...,THREAT INTELLIGENCE REPORT\n\nSEVERITY: CRITIC...
3,You are a cybersecurity threat intelligence an...,CVE ID: CVE-2000-1218\nDescription: The defaul...,THREAT INTELLIGENCE REPORT\n\nSEVERITY: CRITIC...
4,You are a cybersecurity threat intelligence an...,CVE ID: CVE-2000-0944\nDescription: CGI Script...,THREAT INTELLIGENCE REPORT\n\nSEVERITY: CRITIC...


In [205]:
print(f"Shape: {df.shape}")
print(f"Info: {df.info()}")
print(f"Describe: {df.describe()}")

Shape: (472, 3)
<class 'pandas.DataFrame'>
RangeIndex: 472 entries, 0 to 471
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   instruction  472 non-null    str  
 1   input        472 non-null    str  
 2   output       472 non-null    str  
dtypes: str(3)
memory usage: 11.2 KB
Info: None
Describe:                                               instruction  \
count                                                 472   
unique                                                  2   
top     You are a cybersecurity threat intelligence an...   
freq                                                  471   

                                                    input  \
count                                                 472   
unique                                                472   
top     CVE ID: CVE-2001-1481\nDescription: Xitami 2.4...   
freq                                                    1   

                     

In [206]:
print(df.loc[0, "instruction"])
print(df.loc[0, "input"])
print(df.loc[0, "output"][:1000])

You are a cybersecurity threat intelligence analyst. Analyze the following vulnerability data and generate a comprehensive, structured Threat Intelligence Report with severity assessment, technical analysis, MITRE ATT&CK mapping, remediation steps, and detection rules.
CVE ID: CVE-2001-1481
Description: Xitami 2.4 through 2.5 b4 stores the Administrator password in plaintext in the default.aut file, whose default permissions are world-readable, which allows remote attackers to gain privileges.
CVSS Score: 9.8
Severity: CRITICAL
Affected Products: cpe:2.3:a:xitami:xitami:*:*:*:*:*:*:*:*, cpe:2.3:a:xitami:xitami:2.5:beta4:*:*:*:*:*:*
References: http://archives.neohapsis.com/archives/win2ksecadvice/2000-q4/0109.html, http://www.securityfocus.com/archive/1/242375, http://www.securityfocus.com/bid/3582, https://exchange.xforce.ibmcloud.com/vulnerabilities/7600, http://archives.neohapsis.com/archives/win2ksecadvice/2000-q4/0109.html
THREAT INTELLIGENCE REPORT

SEVERITY: CRITICAL (CVSS 9.8)


Threat Categories:
- Remote Code Execution
- SQL Injection
- Cross-Site Scripting
- Credential Exposure
- Privilege Escalation
- Denial of Service
- Information Disclosure
- Directory Traversal
- Authentication Bypass
- Unknown

In [207]:
df.isnull().sum()

instruction    0
input          0
output         0
dtype: int64

In [208]:
for i in range(5):
    print("ROW:", i)
    print(df.loc[i, "input"])
    print("-" * 80)

ROW: 0
CVE ID: CVE-2001-1481
Description: Xitami 2.4 through 2.5 b4 stores the Administrator password in plaintext in the default.aut file, whose default permissions are world-readable, which allows remote attackers to gain privileges.
CVSS Score: 9.8
Severity: CRITICAL
Affected Products: cpe:2.3:a:xitami:xitami:*:*:*:*:*:*:*:*, cpe:2.3:a:xitami:xitami:2.5:beta4:*:*:*:*:*:*
References: http://archives.neohapsis.com/archives/win2ksecadvice/2000-q4/0109.html, http://www.securityfocus.com/archive/1/242375, http://www.securityfocus.com/bid/3582, https://exchange.xforce.ibmcloud.com/vulnerabilities/7600, http://archives.neohapsis.com/archives/win2ksecadvice/2000-q4/0109.html
--------------------------------------------------------------------------------
ROW: 1
CVE ID: CVE-1999-0426
Description: The default permissions of /dev/kmem in Linux versions before 2.0.36 allows IP spoofing.
CVSS Score: 9.8
Severity: CRITICAL
Affected Products: cpe:2.3:o:suse:suse_linux:6.0:*:*:*:*:*:*:*
References:

# Threat Detection
Takes somesort of raw input/data and determines if it is a threat
Am I affected?
How am I affected?

In [209]:

model: SentenceTransformer = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

class ThreatClassificationAgent:
    def __init__(self) -> None:
        self.threats: Dict[str, str] = {
    "SQL Injection": "vulnerability allowing attackers to manipulate SQL database queries through unsanitised user input",
    "Cross-site Scripting (XSS)": "cross-site scripting vulnerability involving malicious script injection and execution in a user's browser",
    "Denial of Service (DOS)": "Denial of Service vulnerabilities involving service exhaustion, malformed packet crashes, server instability, resource exhaustion, or disruption of system availability",
    "Authentication Bypass": "vulnerability allowing attackers to gain unauthorised access without valid authentication credentials, bypass login checks, reset passwords, modify passwords, or access protected functions without proper verification",
    "Privilege Escalation": "vulnerability allowing attackers to gain elevated permissions or administrative access",
    "Remote Code Execution": "vulnerability allowing attackers to remotely execute arbitrary code or system commands on a target machine",
    "Credential Exposure": "vulnerability exposing plaintext passwords, authentication credentials, tokens, or other sensitive login information",
    "Information Disclosure": "vulnerability allowing attackers to access or leak sensitive system, configuration, or user information",
    "Directory Traversal": "vulnerability allowing attackers to access arbitrary files or directories outside intended system paths through path traversal techniques",
    "Brute Force": "vulnerability allowing attackers to repeatedly guess usernames, passwords, or credentials due to weak login protections, missing lockout, or no delay after failed authentication attempts",
    "DNS Poisoning": "vulnerability involving DNS poisoning, spoofed DNS responses, malicious DNS updates, unauthorised name resolution modification, cache poisoning, or redirection of network traffic through manipulated DNS records",
    "Buffer Overflow": "memory corruption vulnerability involving buffer overflow, heap overflow, stack overflow, or overly long input causing code execution, crash, or control-flow manipulation",
    "IP Spoofing": "vulnerability allowing attackers to spoof IP addresses, impersonate trusted hosts, or bypass network-based trust controls"
    }
        self.keys: List[str] = list(self.threats.keys())

    def classify(self, cve_text: str) -> Dict[str, float]:
        input_embeddings: np.array = model.encode(cve_text)
        output_embeddings: np.array = model.encode(list(self.threats.values()))

        input_embeddings: np.array = input_embeddings.reshape(1,-1)
      
        similarity: np.ndarray = cosine_similarity(input_embeddings, output_embeddings)
        one_row: np.ndarray = similarity[0]
        idx: int = one_row.argmax()
        confidence: float = float(one_row[idx])

        if confidence < 0.4:
            return {
                "threat_type": "Unknown",
                "confidence": round(confidence, 3)
            }
        else:
            return {
                "threat_type": self.keys[idx],
                "confidence": round(confidence, 3)
            }    
        
    def __str__(self) -> str:
        return "Threat Classification Agent"

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9377.53it/s]


In [210]:
agent = ThreatClassificationAgent()

x = 4
result = agent.classify(df.loc[x, "input"])
print(result)

{'threat_type': 'Authentication Bypass', 'confidence': 0.465}


In [211]:
agent = ThreatClassificationAgent()

tests = [
    "The application allows attackers to execute arbitrary code remotely.",
    "The login form is vulnerable to SQL injection through unsanitised input.",
    "A reflected cross-site scripting vulnerability allows script execution.",
    "The server crashes when receiving malformed packets, causing denial of service.",
    "Plaintext passwords are stored in a world-readable file."
]

for text in tests:
    print(text)
    print(agent.classify(text))
    print()

The application allows attackers to execute arbitrary code remotely.
{'threat_type': 'Remote Code Execution', 'confidence': 0.698}

The login form is vulnerable to SQL injection through unsanitised input.
{'threat_type': 'SQL Injection', 'confidence': 0.65}

A reflected cross-site scripting vulnerability allows script execution.
{'threat_type': 'Cross-site Scripting (XSS)', 'confidence': 0.839}

The server crashes when receiving malformed packets, causing denial of service.
{'threat_type': 'Denial of Service (DOS)', 'confidence': 0.734}

Plaintext passwords are stored in a world-readable file.
{'threat_type': 'Credential Exposure', 'confidence': 0.508}



In [212]:
for i in range(10):
    text = df.loc[i, "input"]
    print("Row:", i)
    print(agent.classify(text))
    print(text[:300])
    print("-" * 80)

Row: 0
{'threat_type': 'Privilege Escalation', 'confidence': 0.494}
CVE ID: CVE-2001-1481
Description: Xitami 2.4 through 2.5 b4 stores the Administrator password in plaintext in the default.aut file, whose default permissions are world-readable, which allows remote attackers to gain privileges.
CVSS Score: 9.8
Severity: CRITICAL
Affected Products: cpe:2.3:a:xitami:
--------------------------------------------------------------------------------
Row: 1
{'threat_type': 'Unknown', 'confidence': 0.396}
CVE ID: CVE-1999-0426
Description: The default permissions of /dev/kmem in Linux versions before 2.0.36 allows IP spoofing.
CVSS Score: 9.8
Severity: CRITICAL
Affected Products: cpe:2.3:o:suse:suse_linux:6.0:*:*:*:*:*:*:*
References: https://exchange.xforce.ibmcloud.com/vulnerabilities/CVE-1999-0426
--------------------------------------------------------------------------------
Row: 2
{'threat_type': 'Brute Force', 'confidence': 0.446}
CVE ID: CVE-1999-1324
Description: VAXstations running

In [213]:
classification_result = agent.classify(df.loc[8, "input"])
print(classification_result)

{'threat_type': 'Brute Force', 'confidence': 0.525}


# Severity Agent


In [216]:
class SeverityAgent:
    def __init__(self) -> None:
        pass


    def severity_score(self, cve_text, classification_result) -> Dict[str, float]:
        severity_regex: re.Pattern = r'\bSeverity: \b(\w+)'
        cvss_regex: re.Pattern = r'\bCVSS Score: \b([+-]?\d+\.\d+)'

        cvss_score: float = float((re.search(cvss_regex, cve_text)).group(1))
        severity: str  = (re.search(severity_regex, cve_text)).group(1).title()

        threat_type: str = classification_result["threat_type"]
        reason: str = f"The {threat_type} threat has a severity of {severity.lower()} because because it has a CVSS score of {cvss_score}."

        return  {
            "severity": severity,
            "cvss_score": cvss_score,
            "reason": reason,
        }
    
    def __str__(self) -> str:
        return "Severity Agent that identities the severity of a vulnerabilty."

In [217]:
test = df.loc[7, "input"]

agent2 = ThreatClassificationAgent()
severityAgent = SeverityAgent()

classify = agent.classify(test)
severity = severityAgent.severity_score(test, classify)

print(severity)

{'severity': 'Critical', 'cvss_score': 9.8, 'reason': 'The Buffer Overflow threat has a severity of critical because because it has a CVSS score of 9.8.'}


# Mitigation Agents 

In [ ]:
class MitigationAgent:
     def __init__(self):
        pass

# Coordinator Agent

In [ ]:
class CoordinatorAgent:
    def __init__(self):
        pass